# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 8: Machine Learning Prediction & Model Evaluation

In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

data = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv")
X = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv")
Y = data['Class'].to_numpy()

transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print("Test sample records:", Y_test.shape[0])

# Logistic Regression
logreg_cv = GridSearchCV(LogisticRegression(), {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}, cv=10).fit(X_train, Y_train)
# SVM
svm_cv = GridSearchCV(SVC(), {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'), 'C': np.logspace(-3, 3, 5), 'gamma': np.logspace(-3, 3, 5)}, cv=10).fit(X_train, Y_train)
# Decision Tree
tree_cv = GridSearchCV(DecisionTreeClassifier(random_state=2), {'criterion': ['gini', 'entropy'], 'splitter': ['best', 'random'], 'max_depth': [2*n for n in range(1,10)], 'max_features': ['sqrt'], 'min_samples_leaf': [1, 2, 4], 'min_samples_split': [2, 5, 10]}, cv=10).fit(X_train, Y_train)
# KNN
knn_cv = GridSearchCV(KNeighborsClassifier(), {'n_neighbors': list(range(1, 11)), 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 'p': [1, 2]}, cv=10).fit(X_train, Y_train)

print(f"Best SVM kernel: {svm_cv.best_params_['kernel']}")
print(f"LogReg  -> Val: {logreg_cv.best_score_*100:.2f}% | Test: {logreg_cv.score(X_test, Y_test)*100:.2f}%")
print(f"SVM     -> Val: {svm_cv.best_score_*100:.2f}% | Test: {svm_cv.score(X_test, Y_test)*100:.2f}%")
print(f"Tree    -> Val: {tree_cv.best_score_*100:.2f}% | Test: {tree_cv.score(X_test, Y_test)*100:.2f}%")
print(f"KNN     -> Val: {knn_cv.best_score_*100:.2f}% | Test: {knn_cv.score(X_test, Y_test)*100:.2f}%")
print("Confusion Matrix (Test Set):\n", confusion_matrix(Y_test, tree_cv.predict(X_test)))

Test sample records: 18
Best SVM kernel: sigmoid
LogReg  -> Val: 84.64% | Test: 83.33%
SVM     -> Val: 84.82% | Test: 83.33%
Tree    -> Val: 88.93% | Test: 83.33%
KNN     -> Val: 84.82% | Test: 83.33%
Confusion Matrix (Test Set):
 [[ 3  3]
 [ 0 12]]